# Ghost Journal

A small notebook for testing **Ghost on the Shelf** before wiring it to a server.

This journal uses the same chat-message shape as the reference notebook:

```python
messages = [
    {"role": "system", "content": runtime_prompt},
    {"role": "user", "content": "..."}
]
```

It loads the generated runtime prompt and local memory index, retrieves relevant memory fragments, and sends them into the chat request.


## Before running

From the repository root, make sure you have already run:

```bash
uv run --env-file .env python rituals/summarize_runtime.py
uv run --env-file .env python rituals/build_index.py
```

Then start Jupyter from the repository root:

```bash
uv run --env-file .env jupyter lab
```


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
from IPython.display import Markdown, display

import json
import gradio as gr
import numpy as np

In [ ]:
load_dotenv(override=True)
openai = OpenAI()


In [ ]:
ROOT = Path("..")

RUNTIME_PROMPT_PATH = ROOT / "core" / "shelf" / "ghost_runtime.md"
MEMORY_INDEX_PATH = ROOT / "core" / "shelf" / "indexes" / "memory_index.json"

In [ ]:
if not RUNTIME_PROMPT_PATH.exists():
    raise FileNotFoundError(
        "Missing ghost_runtime.md. Run: uv run --env-file .env python rituals/summarize_runtime.py"
    )

if not MEMORY_INDEX_PATH.exists():
    raise FileNotFoundError(
        "Missing memory_index.json. Run: uv run --env-file .env python rituals/build_index.py"
    )

runtime_prompt = RUNTIME_PROMPT_PATH.read_text(encoding="utf-8")

with MEMORY_INDEX_PATH.open("r", encoding="utf-8") as file:
    memory_index = json.load(file)

chunks = memory_index["chunks"]
embedding_model = memory_index["embedding_model"]
embedding_dimensions = memory_index.get("embedding_dimensions")

print(f"Loaded runtime prompt: {len(runtime_prompt.split())} words")
print(f"Loaded memory chunks: {len(chunks)}")
print(f"Embedding model: {embedding_model}")
print(f"Embedding dimensions: {embedding_dimensions}")


In [ ]:
def embed_text(text: str) -> np.ndarray:
    kwargs = {
        "model": embedding_model,
        "input": text,
    }

    if embedding_dimensions:
        kwargs["dimensions"] = embedding_dimensions

    response = openai.embeddings.create(**kwargs)
    return np.array(response.data[0].embedding, dtype=np.float32)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denominator = np.linalg.norm(a) * np.linalg.norm(b)

    if denominator == 0:
        return 0.0

    return float(np.dot(a, b) / denominator)


def retrieve_memory_fragments(question: str, k: int = 4) -> list[dict]:
    question_embedding = embed_text(question)
    scored_chunks = []

    for chunk in chunks:
        chunk_embedding = np.array(chunk["embedding"], dtype=np.float32)
        score = cosine_similarity(question_embedding, chunk_embedding)

        scored_chunks.append(
            {
                "score": score,
                "id": chunk["id"],
                "source": chunk["source"],
                "module": chunk["module"],
                "title": chunk["title"],
                "text": chunk["text"],
            }
        )

    return sorted(scored_chunks, key=lambda item: item["score"], reverse=True)[:k]


def format_memory_fragments(matches: list[dict]) -> str:
    if not matches:
        return "No memory fragments retrieved."

    formatted = []

    for index, match in enumerate(matches, start=1):
        formatted.append(
            f'''## Fragment {index}
Source: {match["source"]}
Module: {match["module"]}
Title: {match["title"]}
Score: {match["score"]:.3f}

{match["text"]}'''
        )

    return "\n\n---\n\n".join(formatted)


In [ ]:
question = "What projects suggest an interest in cognition and learning?"

matches = retrieve_memory_fragments(question, k=4)

for match in matches:
    print(f'{match["score"]:.3f} | {match["module"]} | {match["title"]} | {match["source"]}')


In [ ]:
def build_messages(
    question: str,
    retrieved_context: str,
    session_summary: str = "No prior session summary.",
) -> list[dict]:
    user_content = f'''SESSION SUMMARY:
{session_summary}

RETRIEVED MEMORY FRAGMENTS:
{retrieved_context}

USER MESSAGE:
{question}'''

    return [
        {"role": "system", "content": runtime_prompt},
        {"role": "user", "content": user_content},
    ]


In [ ]:
MODEL = "gpt-5-mini"


def ask_ghost(
    question: str,
    k: int = 4,
    session_summary: str = "No prior session summary.",
    display_fragments: bool = True,
) -> str:
    matches = retrieve_memory_fragments(question, k=k)
    retrieved_context = format_memory_fragments(matches)
    messages = build_messages(question, retrieved_context, session_summary)

    if display_fragments:
        print("Retrieved fragments:")
        for match in matches:
            print(f'- {match["score"]:.3f} | {match["title"]} | {match["source"]}')
        print()

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    answer = response.choices[0].message.content
    display(Markdown(answer))

    return answer


In [ ]:
answer = ask_ghost("What kind of projects do you like?", k=3)

In [ ]:
answer = ask_ghost("What do you think about cute accelerationism?", k=3)

In [ ]:
answer = ask_ghost("What do you think of Thursday Next?", k=2)

In [ ]:
answer = ask_ghost("What do you think of German tax bureaucracy?", k=3)

In [ ]:
answer = ask_ghost("Beauty or fairness? Pick one", k=3)

In [ ]:
answer = ask_ghost("Tell me more about the ghost on the shelf project", k=3)

In [ ]:
answer = ask_ghost("Tell me more about Chibo", k=3)

In [ ]:
answer = ask_ghost("Are you mogging?", k=3)

In [ ]:
answer = ask_ghost("What memory are you mostly proud of?", k=3)

In [ ]:
answer = ask_ghost("What are you an expert on?", k=3)

In [ ]:
answer = ask_ghost("Are you Fey?", k=3)

In [ ]:
answer = ask_ghost("日本語が話せますか", k=3)

In [ ]:
answer = ask_ghost("¿Hablas español?", k=3)

## Notes

Things to tune after testing:

- If answers are too long, reduce retrieved fragments with `k=2` or add stricter brevity to `voice.md`.
- If answers feel generic, add more concrete memories or style examples.
- If retrieval feels wrong, inspect the printed fragments and edit the relevant `Signals:` sections.
- If drift mode is too dramatic, soften the runtime prompt in `voice.md`.
